In [1]:
import polars as pl
import points_dict
from tournament_dict import tournaments

In [26]:
# read file
file_2024 = 'atp_matches_2024.csv'
df = pl.read_csv(file_2024, infer_schema_length=None)
print(df.shape) # (3076, 49)

(3076, 49)


In [3]:
keep_cols = ['tourney_name', 'surface', 'draw_size', 'tourney_level', 'tourney_date', 'winner_id', 'winner_name', 'winner_seed', 'loser_id', 'loser_name', 'loser_seed', 'round']
winner_cols_rename = {'winner_id':'player_id', 'winner_name':'player_name', 'winner_seed':'player_seed'} 
non_winner_cols_rename = {'loser_id':'player_id', 'loser_name':'player_name', 'loser_seed':'player_seed'}

df1 = df[keep_cols].clone()
df1.shape

(3076, 12)

In [31]:
def get_points(df, tourney_level, draw_size, points_dict):
    df_filter = df.filter((pl.col('tourney_level') == tourney_level) & (pl.col('draw_size') == draw_size))

    df_non_winner = df_filter.with_columns(pl.col('round').replace_strict(points_dict, default=None).alias('points'))
    df_non_winner = df_non_winner[['tourney_name','tourney_level','loser_id','loser_name','loser_seed','round','points']].rename(non_winner_cols_rename)

    df_winner = (
        df_filter
        .filter(pl.col('round') == 'F')
        .with_columns([
            pl.lit('W').alias('round'),
            pl.lit(points_dict['W']).alias('points')
        ])
    )

    df_winner = df_winner[['tourney_name','tourney_level','winner_id','winner_name','winner_seed','round','points']].rename(winner_cols_rename)
    df_winner

    df_non_winner = df_non_winner.with_columns(pl.col('points').cast(pl.Int32))
    df_points = pl.concat([df_non_winner, df_winner])
    df_points

    return df_points

In [42]:
df_gs = get_points(df1, 'G', 128, points_dict.points_GS)

In [ ]:
# df_gs = df1.filter(pl.col('tourney_level')=='G')

# df_gs_non_winner = df_gs.with_columns(pl.col('round').replace_strict(points_dict.points_GS, default=None).alias('points'))
# df_gs_non_winner = df_gs_non_winner[['tourney_name','tourney_level','loser_id','loser_name','points']].rename(non_winner_cols_rename)

# df_gs_winner = (
#     df_gs
#     .filter(pl.col('round') == 'F')
#     .with_columns([
#         pl.lit('W').alias('round'),
#         pl.lit(points_dict.points_GS['W']).alias('points')
#     ])
# )
# df_gs_winner = df_gs_winner[['tourney_name','tourney_level','winner_id','winner_name','points']].rename(winner_cols_rename)
# df_gs_winner

# df_gs_non_winner = df_gs_non_winner.with_columns(pl.col('points').cast(pl.Int32))
# df_gs_points = pl.concat([df_gs_non_winner, df_gs_winner])
# df_gs_points

tourney_name,tourney_level,player_id,player_name,points
str,str,i64,str,i32
"""Australian Open""","""G""",209976,"""Dino Prizmic""",10
"""Australian Open""","""G""",117360,"""Marc Polmans""",10
"""Australian Open""","""G""",105870,"""Yannick Hanfmann""",10
"""Australian Open""","""G""",104918,"""Andy Murray""",10
"""Australian Open""","""G""",104527,"""Stan Wawrinka""",10
…,…,…,…,…
"""Us Open""","""G""",126203,"""Taylor Fritz""",1300
"""Australian Open""","""G""",206173,"""Jannik Sinner""",2000
"""Roland Garros""","""G""",207989,"""Carlos Alcaraz""",2000


In [47]:
df_gs_points = df_gs.pivot(
    values = "points",
    index = ["player_id", "player_name"],
    on = ["tourney_name", "tourney_level"],
    aggregate_function="first"
)

df_gs_points.filter(pl.col("player_name").is_in(['Carlos Alcaraz','Jannik Sinner','Novak Djokovic']))

player_id,player_name,"{""Australian Open"",""G""}","{""Roland Garros"",""G""}","{""Wimbledon"",""G""}","{""Us Open"",""G""}"
i64,str,i32,i32,i32,i32
207989,"""Carlos Alcaraz""",400,2000,2000,50
104925,"""Novak Djokovic""",800,400,1300,100
206173,"""Jannik Sinner""",2000,800,400,2000


In [ ]:
### ATP Masters ###
df_masters = df1.filter(pl.col('tourney_level')=='M')
df_masters['tourney_name'].value_counts()

In [21]:
### 250 and 500 with 32 player draw ###
df_atp = df1.filter((pl.col('tourney_level')=='A') & (~pl.col('tourney_name').is_in(['United Cup', 'Laver Cup'])))

df_atp = df_atp.with_columns(pl.col("tourney_name").replace_strict(tournaments).alias('tourney_info'))

df_atp = df_atp.with_columns(
    pl.col("tourney_info").list.get(0).alias("tourney_level"),
    pl.col("tourney_info").list.get(1).alias("tourney_name") # -1 gets the last element
)

df_atp_250 = df_atp.filter(pl.col('tourney_level') == '250')


(1081, 13) (404, 13)


In [37]:
get_points(df_atp, '500', 64, points_dict.points_500_48)

tourney_name,tourney_level,player_id,player_name,player_seed,round,points
str,str,i64,str,i64,str,i32
"""Barcelona Open""","""500""",126774,"""Stefanos Tsitsipas""",5,"""F""",330
"""Barcelona Open""","""500""",105583,"""Dusan Lajovic""",null,"""SF""",200
"""Barcelona Open""","""500""",144869,"""Tomas Martin Etcheverry""",13,"""SF""",200
"""Barcelona Open""","""500""",207680,"""Facundo Diaz Acosta""",null,"""QF""",100
"""Barcelona Open""","""500""",209950,"""Arthur Fils""",16,"""QF""",100
…,…,…,…,…,…,…
"""Washington Open""","""500""",208260,"""Zachary Svajda""",null,"""R64""",0
"""Washington Open""","""500""",208004,"""Harold Mayot""",null,"""R64""",0
"""Washington Open""","""500""",207680,"""Facundo Diaz Acosta""",null,"""R64""",0


In [ ]:
### ATP 500 - 32 Draw ###
df_atp_500 = df_atp.filter(pl.col('tourney_level') == '500')

tourney_name,surface,draw_size,tourney_level,tourney_date,winner_id,winner_name,winner_seed,loser_id,loser_name,loser_seed,round,tourney_info,points
str,str,i64,str,i64,i64,str,i64,i64,str,i64,str,list[str],i64
"""Barcelona Open""","""Clay""",64,"""500""",20240415,134770,"""Casper Ruud""",3,126774,"""Stefanos Tsitsipas""",5,"""F""","[""500"", ""Barcelona Open""]",330
"""Barcelona Open""","""Clay""",64,"""500""",20240415,126774,"""Stefanos Tsitsipas""",5,105583,"""Dusan Lajovic""",null,"""SF""","[""500"", ""Barcelona Open""]",200
"""Barcelona Open""","""Clay""",64,"""500""",20240415,134770,"""Casper Ruud""",3,144869,"""Tomas Martin Etcheverry""",13,"""SF""","[""500"", ""Barcelona Open""]",200
"""Barcelona Open""","""Clay""",64,"""500""",20240415,126774,"""Stefanos Tsitsipas""",5,207680,"""Facundo Diaz Acosta""",null,"""QF""","[""500"", ""Barcelona Open""]",100
"""Barcelona Open""","""Clay""",64,"""500""",20240415,105583,"""Dusan Lajovic""",null,209950,"""Arthur Fils""",16,"""QF""","[""500"", ""Barcelona Open""]",100
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Washington Open""","""Hard""",64,"""500""",20240729,111805,"""Seong Chan Hong""",null,200443,"""Adam Walton""",null,"""R64""","[""500"", ""Washington Open""]",null
"""Washington Open""","""Hard""",64,"""500""",20240729,133430,"""Denis Shapovalov""",null,105138,"""Roberto Bautista Agut""",null,"""R64""","[""500"", ""Washington Open""]",null
"""Washington Open""","""Hard""",64,"""500""",20240729,200670,"""J J Wolf""",null,208260,"""Zachary Svajda""",null,"""R64""","[""500"", ""Washington Open""]",null


In [ ]:
### ATP 500 - 64 Draw ###
df_atp_500_64 = df_atp_500.filter(pl.col('draw_size')==64).with_columns(pl.col("round").replace_strict(points_dict.points_500_32, default=None).alias("points"))
